# Week 5 &mdash; Manifolds and Riemannian Geometry

## Theory: Tangent Spaces, the Laplace&ndash;Beltrami Operator, and Gradient Flow

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

### Learning Objectives

1. Define a **smooth manifold** and its **tangent space**, and the role of charts.
2. Introduce a **Riemannian metric** and the geometric quantities it induces (length, gradient, geodesics).
3. Define the **Laplace&ndash;Beltrami operator** as the intrinsic generalisation of the Laplacian.
4. Connect the continuous operator to the **discrete cotangent Laplacian** on meshes.
5. Understand **gradient flow** / **heat diffusion** on manifolds and its role in geometric learning.

---


## 1. Smooth Manifolds and Tangent Spaces

Grids and graphs are *combinatorial* domains; surfaces of 3D objects are *continuous, curved* domains. The mathematical model is the **manifold**.

A **smooth $n$-manifold** $\mathcal{M}$ is a space that locally resembles $\mathbb{R}^n$: every point has a neighbourhood mapped homeomorphically to an open subset of $\mathbb{R}^n$ by a **chart** $\varphi : U \to \mathbb{R}^n$, with smooth transitions between overlapping charts. A surface in $\mathbb{R}^3$ (e.g. the boundary of a 3D shape) is a 2-manifold.

At each point $p \in \mathcal{M}$, the **tangent space** $T_p\mathcal{M}$ is the $n$-dimensional vector space of velocities of curves through $p$. Intuitively, it is the *best linear (flat) approximation* to the manifold at $p$: for a surface in $\mathbb{R}^3$, $T_p\mathcal{M}$ is the tangent plane. The disjoint union of all tangent spaces is the **tangent bundle** $T\mathcal{M}$; a **vector field** is a smooth choice of tangent vector at each point.

The tangent space is where calculus on the manifold happens: gradients, directional derivatives, and the construction of equivariant filters (gauge-equivariant CNNs) all live in $T_p\mathcal{M}$.


## 2. The Riemannian Metric

A bare smooth manifold has no notion of length or angle. A **Riemannian metric** $g$ supplies one: it assigns to each point $p$ an inner product $g_p(\cdot,\cdot)$ on $T_p\mathcal{M}$, varying smoothly with $p$. In a chart, $g$ is a symmetric positive-definite matrix field $g_{ij}(p)$.

From the metric flow all intrinsic geometric quantities:

- **Length of a curve** $\gamma : [0,1]\to\mathcal{M}$: $\;L(\gamma) = \int_0^1 \sqrt{g_{\gamma(t)}(\dot\gamma, \dot\gamma)}\; dt.$
- **Geodesic distance** $d(p,q)$: the infimum of lengths over curves joining $p$ and $q$. Geodesics generalise straight lines.
- **Volume element** $dV = \sqrt{\det g}\; dx^1 \cdots dx^n$, needed to integrate functions on $\mathcal{M}$.
- **Gradient** of a scalar field $f$: the unique tangent vector $\nabla f$ with $g(\nabla f, v) = df(v)$ for all $v$.

The metric is precisely the structure preserved by the relevant symmetry group of a manifold: its **isometries**. Geometric learning on manifolds seeks operators that are equivariant to isometries &mdash; the Week 1 blueprint, now on a curved domain.


## 3. The Laplace&ndash;Beltrami Operator

The single most important operator in manifold learning is the **Laplace&ndash;Beltrami operator** $\Delta_{\mathcal{M}}$, the generalisation of the Euclidean Laplacian to curved domains. For a scalar field $f$,
$$
\Delta_{\mathcal{M}} f = \operatorname{div}(\nabla f) = \frac{1}{\sqrt{\det g}}\,\partial_i\!\Big(\sqrt{\det g}\; g^{ij}\, \partial_j f\Big),
$$
where $g^{ij}$ is the inverse metric (Einstein summation implied). When $\mathcal{M} = \mathbb{R}^n$ with the Euclidean metric, this reduces to the familiar $\sum_i \partial_i^2 f$.

**Why it is central.**

- It is **intrinsic**: defined purely from the metric, hence **invariant to isometries** (bending the surface without stretching leaves $\Delta_{\mathcal{M}}$ unchanged). Features built from it are automatically isometry-invariant &mdash; ideal for non-rigid shape analysis.
- It is **self-adjoint and negative semi-definite** with respect to the $L^2$ inner product induced by $dV$, so it admits an orthonormal eigenbasis
$$
\Delta_{\mathcal{M}} \phi_k = -\lambda_k \phi_k, \qquad 0 = \lambda_0 \le \lambda_1 \le \cdots,
$$
the **manifold Fourier basis** &mdash; the continuous analogue of the graph Fourier modes of Week 2. This makes *spectral* methods (shape descriptors, spectral filtering on meshes) possible.


## 4. Discretisation: The Cotangent Laplacian

To compute on a triangle mesh $\mathcal{M} \approx (V, E, F)$ we need a discrete $\Delta_{\mathcal{M}}$. The standard choice is the **cotangent Laplacian**, derived from a finite-element / discrete-exterior-calculus discretisation:
$$
(L f)_i = \frac{1}{2 A_i} \sum_{j \in \mathcal{N}(i)} \big(\cot \alpha_{ij} + \cot \beta_{ij}\big)\,(f_i - f_j),
$$
where $\alpha_{ij}, \beta_{ij}$ are the two angles opposite the edge $(i,j)$ and $A_i$ is the (Voronoi or barycentric) area associated with vertex $i$.

In matrix form $L = M^{-1} C$, with $C$ the symmetric **cotangent weight** (stiffness) matrix and $M$ the diagonal **mass matrix** of vertex areas. This is the natural geometric generalisation of the graph Laplacian $L = D - A$ from Week 2: the *combinatorial* weights $A_{ij}$ are replaced by *geometric* weights $\tfrac{1}{2}(\cot\alpha_{ij} + \cot\beta_{ij})$ that account for the mesh's shape, and the degree normalisation by the mass matrix $M$. As the mesh is refined, $L$ converges to the smooth $\Delta_{\mathcal{M}}$.


## 5. Gradient Flow and Heat Diffusion

Many geometric algorithms are **flows**: continuous-time evolutions driven by a gradient or a differential operator.

**Heat diffusion.** The most important is the **heat equation** on $\mathcal{M}$,
$$
\frac{\partial u}{\partial t} = \Delta_{\mathcal{M}}\, u, \qquad u(\cdot, 0) = u_0,
$$
whose solution $u(\cdot, t) = e^{t\Delta_{\mathcal{M}}} u_0$ smooths the initial signal at a rate set by the geometry. The associated **heat kernel** $h_t(p,q)$ encodes how heat propagates between points and is the basis of powerful isometry-invariant descriptors such as the **Heat Kernel Signature (HKS)** and the **Global Point Signature**.

**Gradient flow.** More generally, minimising an energy $E(u)$ defined on fields over $\mathcal{M}$ by steepest descent,
$$
\frac{\partial u}{\partial t} = -\nabla_{L^2} E(u),
$$
yields a gradient flow. The heat equation is itself the gradient flow of the **Dirichlet energy** $E(u) = \tfrac{1}{2}\int_{\mathcal{M}} \|\nabla u\|^2\, dV$ &mdash; exactly the continuous analogue of the graph Dirichlet energy of Week 2.

**Relevance to deep learning.** These flows motivate modern architectures. **Diffusion-based GNNs** (e.g. GRAND) interpret message passing as discretised heat diffusion on a graph or mesh; **spectral mesh CNNs** filter signals in the $\Delta_{\mathcal{M}}$ eigenbasis; and intrinsic descriptors from heat flow provide isometry-invariant features for the shape-segmentation task of Week 6.


## 6. Summary

- A **manifold** models curved domains; the **tangent space** $T_p\mathcal{M}$ is its local linearisation and the home of calculus.
- A **Riemannian metric** $g$ defines lengths, geodesics, volume, and the gradient.
- The **Laplace&ndash;Beltrami operator** $\Delta_{\mathcal{M}}$ is the intrinsic, isometry-invariant Laplacian; its eigenbasis is the manifold Fourier basis.
- On meshes it is discretised by the **cotangent Laplacian** $L = M^{-1}C$, generalising the graph Laplacian.
- **Heat diffusion** and **gradient flows** generated by $\Delta_{\mathcal{M}}$ underpin spectral descriptors and diffusion-based geometric networks.

### References

- Bronstein, M. M., Bruna, J., Cohen, T., &amp; Veli&#269;kovi&#263;, P. (2021). *Geometric Deep Learning.* arXiv:2104.13478, Chs. on Geodesics &amp; Gauges.
- do Carmo, M. P. (1992). *Riemannian Geometry.* Birkh&auml;user.
- Sun, J., Ovsjanikov, M., &amp; Guibas, L. (2009). *A Concise and Provably Informative Multi-Scale Signature Based on Heat Diffusion.* SGP.
- Chamberlain, B., et al. (2021). *GRAND: Graph Neural Diffusion.* ICML.

### Exercises

1. Verify that $\Delta_{\mathcal{M}}$ reduces to $\sum_i \partial_i^2$ when $g = I$.
2. Show that the heat equation is the $L^2$ gradient flow of the Dirichlet energy.
3. Explain why $\Delta_{\mathcal{M}}$-based descriptors are invariant to *isometric* (non-rigid) deformations but a raw coordinate-based descriptor is not.
